In [1]:
# 5_case_categories.py
# Success and failure categories of class-conditional (Mondrian) conformal prediction at the 90% target, judged against argmax.
# Run after 1_code.ipynb. Reads modeling_table.csv and core_outputs/ (egd_probas.npy, utsw_probas.npy, is_test_egd.npy, is_test_utsw.npy).
# Writes core_outputs/case_categories_long.csv    (one row per patient per split in which the patient was in the test half)
#        core_outputs/case_categories_patients.csv (one row per patient: most common category and set, and how often they occurred)
#        core_outputs/table2_categories.csv        (Table 2: share of test patients in each category, mean over the 200 splits)
#
# Three questions per test patient: was argmax right, is the true subtype in the 90% set, and how many subtypes does the set name.
# Argmax is "confident" when its top probability is at least 0.80 (section 1 shows argmax is right in at least 90% of patients from there).
# Every test patient falls in exactly one category:
#   Successes
#   S1 Confident and correct               argmax right, set = {truth}
#   S2 Borderline call, ambiguity flagged  argmax right but borderline (top probability < 0.80), set = truth + one other subtype
#   S3 Wrong call caught                   argmax wrong, set = truth + one other subtype
#   S4 Wrong call overridden               argmax wrong, set = {truth}
#   Failures
#   F1 Correct call, unnecessary hedging   argmax right and confident, set = truth + one other subtype
#   F2 Valid but uninformative             set = all three subtypes (argmax right or wrong)
#   F3 Excludes the correct call           argmax right, truth not in the set (empty sets included)
#   F4 Wrong call not corrected            argmax wrong, truth not in the set (empty sets included)
# A = astrocytoma, G = glioblastoma, O = oligodendroglioma.

import os
import numpy as np
import pandas as pd
from itertools import product
from crepes import ConformalClassifier

OUT = 'core_outputs'
table = pd.read_csv('modeling_table.csv', usecols=['case_id', 'site', '2021_glioma_class'])
classes = np.array(['astrocytoma', 'glioblastoma', 'oligodendroglioma'])
letters = np.array(['A', 'G', 'O'])
level = 0.90
confident = 0.80

d_egd = table[table['site'] == 'EGD'].reset_index(drop=True)
d_utsw = table[table['site'] == 'UTSW'].reset_index(drop=True)
cohorts = {'EGD': d_egd, 'UTSW': d_utsw}
probas = {'EGD': np.load(os.path.join(OUT, 'egd_probas.npy')), 'UTSW': np.load(os.path.join(OUT, 'utsw_probas.npy'))}
is_test = {'EGD': np.load(os.path.join(OUT, 'is_test_egd.npy')), 'UTSW': np.load(os.path.join(OUT, 'is_test_utsw.npy'))}

# ---- 1. argmax accuracy by top probability, both external cohorts, one value per patient (justifies the 0.80 cutoff) ----
p_all = np.vstack([probas['EGD'], probas['UTSW']])
y_all = np.concatenate([np.searchsorted(classes, d_egd['2021_glioma_class'].values), np.searchsorted(classes, d_utsw['2021_glioma_class'].values)])
top_all = p_all.max(axis=1)
right_all = np.argmax(p_all, axis=1) == y_all
for cut in [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90]:
    keep = top_all >= cut
    print('top probability >= {:.2f}: argmax right in {:.1f}% ({:.0f}% of patients)'.format(cut, right_all[keep].mean() * 100, keep.mean() * 100))

# ---- 2. category of every test patient in every split (same Mondrian fit as 1_code.ipynb) ----
split_rows = []
for cohort, seed in product(['EGD', 'UTSW'], range(200)):
    d = cohorts[cohort]
    y = d['2021_glioma_class'].values
    col = np.searchsorted(classes, y)
    p = probas[cohort]
    scores = 1 - p
    test = np.where(is_test[cohort][seed])[0]
    cal = np.where(~is_test[cohort][seed])[0]
    cc = ConformalClassifier()
    cc.fit(scores[cal, col[cal]], bins=y[cal])
    sets = np.column_stack([cc.predict_set(scores[test], bins=np.full(len(test), cls), confidence=level, smoothing=False)[:, k]
                            for k, cls in enumerate(classes)])
    yt = col[test]
    size = sets.sum(axis=1)
    covered = sets[np.arange(len(test)), yt] == 1
    argmax = np.argmax(p[test], axis=1)
    right = argmax == yt
    sure = p[test].max(axis=1) >= confident

    # one condition per category; together they cover every patient exactly once
    conditions = [right & covered & (size == 1),
                  right & covered & (size == 2) & (~sure),
                  (~right) & covered & (size == 2),
                  (~right) & covered & (size == 1),
                  right & covered & (size == 2) & sure,
                  covered & (size == 3),
                  right & (~covered),
                  (~right) & (~covered)]
    names = ['S1 Confident and correct', 'S2 Borderline call, ambiguity flagged', 'S3 Wrong call caught', 'S4 Wrong call overridden',
             'F1 Correct call, unnecessary hedging', 'F2 Valid but uninformative', 'F3 Excludes the correct call', 'F4 Wrong call not corrected']
    category = np.select(conditions, names, default='unassigned')

    # the set written as text, e.g. {A, O}
    set_text = ['{' + ', '.join(letters[sets[i] == 1]) + '}' for i in range(len(test))]

    split_rows.append(pd.DataFrame({'cohort': cohort, 'seed': seed, 'case_id': d['case_id'].values[test],
                                    'true': letters[yt], 'argmax': letters[argmax], 'argmax_correct': right,
                                    'top_probability': p[test].max(axis=1), 'set': set_text, 'category': category}))

long = pd.concat(split_rows, ignore_index=True)
long.to_csv(os.path.join(OUT, 'case_categories_long.csv'), index=False)
print('\nunassigned rows (must be 0):', np.sum(long['category'] == 'unassigned'))

# ---- 3. Table 2: share of test patients in each category, mean and 2.5th-97.5th percentile over the 200 splits ----
counts = long.groupby(['cohort', 'seed', 'category']).size().unstack(fill_value=0)
shares = counts.div(counts.sum(axis=1), axis=0) * 100
mean_share = shares.groupby('cohort').mean().T
low_share = shares.groupby('cohort').quantile(0.025).T
high_share = shares.groupby('cohort').quantile(0.975).T
composition = pd.crosstab([long['cohort'], long['category']], long['true'], normalize='index') * 100

table2 = pd.DataFrame({'EGD, % of test patients': mean_share['EGD'].round(1),
                       'EGD, range': low_share['EGD'].round(1).astype(str) + '-' + high_share['EGD'].round(1).astype(str),
                       'UTSW-Glioma, % of test patients': mean_share['UTSW'].round(1),
                       'UTSW-Glioma, range': low_share['UTSW'].round(1).astype(str) + '-' + high_share['UTSW'].round(1).astype(str)})
table2.to_csv(os.path.join(OUT, 'table2_categories.csv'))
print(table2)
print('\ntrue subtype within each category (%):')
print(composition.round(0))

# ---- 4. one row per patient: most common category and set, and their share of the patient's test-half splits ----
patients = long.groupby(['cohort', 'case_id']).agg(true=('true', 'first'), argmax=('argmax', 'first'),
                                                   argmax_correct=('argmax_correct', 'first'), n_test=('seed', 'size')).reset_index()
modal_cat = long.groupby(['cohort', 'case_id'])['category'].agg(lambda s: s.value_counts().index[0]).rename('modal_category')
cat_share = long.groupby(['cohort', 'case_id'])['category'].agg(lambda s: s.value_counts().iloc[0] / len(s)).rename('category_stability')
modal_set = long.groupby(['cohort', 'case_id'])['set'].agg(lambda s: s.value_counts().index[0]).rename('modal_set')
set_share = long.groupby(['cohort', 'case_id'])['set'].agg(lambda s: s.value_counts().iloc[0] / len(s)).rename('set_stability')
patients = patients.join(modal_cat, on=['cohort', 'case_id']).join(cat_share, on=['cohort', 'case_id'])
patients = patients.join(modal_set, on=['cohort', 'case_id']).join(set_share, on=['cohort', 'case_id'])

prob_egd = pd.DataFrame({'case_id': d_egd['case_id'], 'pA': probas['EGD'][:, 0], 'pG': probas['EGD'][:, 1], 'pO': probas['EGD'][:, 2]})
prob_utsw = pd.DataFrame({'case_id': d_utsw['case_id'], 'pA': probas['UTSW'][:, 0], 'pG': probas['UTSW'][:, 1], 'pO': probas['UTSW'][:, 2]})
patients = patients.merge(pd.concat([prob_egd, prob_utsw]), on='case_id')
patients.to_csv(os.path.join(OUT, 'case_categories_patients.csv'), index=False)

# ---- 5. candidate examples: same category in at least 90% of the patient's test-half splits ----
stable = patients[patients['category_stability'] >= 0.90]
print('\nstable patients per category:')
print(stable.groupby('modal_category').size())

# the patients used in Figures 5 and 6
examples = ['BT1106', 'EGD-0106', 'EGD-0022', 'EGD-0368', 'EGD-0026', 'BT0438', 'BT0291', 'BT0011']
print(patients[patients['case_id'].isin(examples)].round(2).to_string(index=False))


top probability >= 0.50: argmax right in 81.4% (96% of patients)
top probability >= 0.60: argmax right in 84.7% (87% of patients)
top probability >= 0.70: argmax right in 88.0% (80% of patients)
top probability >= 0.75: argmax right in 89.9% (75% of patients)
top probability >= 0.80: argmax right in 91.6% (70% of patients)
top probability >= 0.85: argmax right in 93.9% (63% of patients)
top probability >= 0.90: argmax right in 95.9% (50% of patients)

unassigned rows (must be 0): 0
                                       EGD, % of test patients EGD, range  \
category                                                                    
F1 Correct call, unnecessary hedging                      10.3   4.0-17.3   
F2 Valid but uninformative                                 0.8    0.0-5.5   
F3 Excludes the correct call                               3.3    0.5-7.4   
F4 Wrong call not corrected                                5.4    2.0-9.4   
S1 Confident and correct                           

In [2]:
import os
os.getcwd()

'g:\\My Drive\\bsd\\11_radiomics_for_gliomas\\02_train\\trial_100\\AJNR submission\\revision1\\code4'

In [1]:
# 5_case_figures.py
# Figure 5 (successes S1-S4) and Figure 6 (failures F1-F4) of class-conditional conformal prediction against argmax, 90% target.
# Run after 5_case_categories.py. Reads core_outputs/case_categories_patients.csv and table2_categories.csv
# and, for each example patient, t1ce_norm.nii.gz, flair_norm.nii.gz and seg.nii.gz from the preprocessed folder.
# Writes core_outputs/Figure5.png/.tiff and Figure6.png/.tiff
# Each row: category and how often it occurs (left), contrast-enhanced T1 and FLAIR at the axial slice with the largest tumor area,
# whole-tumor outline in yellow (middle), probabilities, argmax and the 90% set (right).
# Images: reoriented to RAS, shown anterior up in radiological convention (patient right on image left), window 1st-99.5th
# percentile of brain voxels on that slice, cropped to the brain.
# Examples: same category in at least 90% of the splits in which the patient was in the test half (5_case_categories.py, section 5).

import os
import textwrap
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

OUT = 'core_outputs'
IMG = '../../../../trial_80/preprocessed'
patients = pd.read_csv(os.path.join(OUT, 'case_categories_patients.csv')).set_index('case_id')
table2 = pd.read_csv(os.path.join(OUT, 'table2_categories.csv')).set_index('category')
full_name = {'A': 'Astrocytoma', 'G': 'Glioblastoma', 'O': 'Oligodendroglioma'}
plt.rcParams.update({'font.family': ['Arial', 'Liberation Sans', 'DejaVu Sans'], 'font.size': 7})

# one row per panel: figure, letter, category (as in table2_categories.csv), description, patient
panels = [
    ('Figure5', 'A', 'S1 Confident and correct', 'Argmax is right, and the set names only the true subtype.', 'BT1106'),
    ('Figure5', 'B', 'S2 Borderline call, ambiguity flagged', 'Argmax is right but borderline (top probability below 0.80). The set adds the close second subtype.', 'EGD-0106'),
    ('Figure5', 'C', 'S3 Wrong call caught', 'Argmax is wrong. The set keeps the true subtype alongside the wrong call.', 'EGD-0022'),
    ('Figure5', 'D', 'S4 Wrong call overridden', 'Argmax is wrong. The set leaves out the wrong call and names the true subtype alone.', 'EGD-0368'),
    ('Figure6', 'A', 'F1 Correct call, unnecessary hedging', 'Argmax is confident and right, but the set adds a second subtype.', 'EGD-0026'),
    ('Figure6', 'B', 'F2 Valid but uninformative', 'The set contains the true subtype but excludes nothing.', 'BT0438'),
    ('Figure6', 'C', 'F3 Excludes the correct call', 'Argmax is right, but the glioblastoma probability is below its cutoff, so the set leaves the true subtype out.', 'BT0291'),
    ('Figure6', 'D', 'F4 Wrong call not corrected', 'Argmax is wrong, and the set names the same wrong subtype.', 'BT0011')]

rows_per_fig = {'Figure5': 4, 'Figure6': 4}
figs = {}
grids = {}
row_index = {'Figure5': 0, 'Figure6': 0}
for name in ['Figure5', 'Figure6']:
    n = rows_per_fig[name]
    height = 1.95 * n + 0.35
    figs[name] = plt.figure(figsize=(7.4, height))
    grids[name] = figs[name].add_gridspec(n, 4, width_ratios=[1.35, 1.25, 1.25, 1.55], wspace=0.06, hspace=0.12,
                                          left=0.01, right=0.99, top=1 - 0.3 / height, bottom=0.01)

for fig_name, letter, category, description, case in panels:
    fig = figs[fig_name]
    gs = grids[fig_name]
    i = row_index[fig_name]
    row_index[fig_name] = i + 1
    r = patients.loc[case]
    folder = os.path.join(IMG, 'MR_' + case if case.startswith('EGD') else case)

    # load and reorient to RAS
    t1ce = np.asarray(nib.as_closest_canonical(nib.load(os.path.join(folder, 't1ce_norm.nii.gz'))).dataobj, dtype=float)
    flair = np.asarray(nib.as_closest_canonical(nib.load(os.path.join(folder, 'flair_norm.nii.gz'))).dataobj, dtype=float)
    seg = np.asarray(nib.as_closest_canonical(nib.load(os.path.join(folder, 'seg.nii.gz'))).dataobj) > 0

    # axial slice with the largest tumor area, anterior up, patient right on image left
    k = int(np.argmax(seg.sum(axis=(0, 1))))
    t1ce_slice = np.fliplr(np.rot90(t1ce[:, :, k]))
    flair_slice = np.fliplr(np.rot90(flair[:, :, k]))
    mask_slice = np.fliplr(np.rot90(seg[:, :, k])).astype(float)
    brain_slice = (t1ce_slice != 0) | (flair_slice != 0)

    # crop to the brain with a 6-pixel margin
    ys, xs = np.where(brain_slice)
    y0 = max(ys.min() - 6, 0)
    y1 = ys.max() + 6
    x0 = max(xs.min() - 6, 0)
    x1 = xs.max() + 6
    t1ce_crop = t1ce_slice[y0:y1, x0:x1]
    flair_crop = flair_slice[y0:y1, x0:x1]
    mask_crop = mask_slice[y0:y1, x0:x1]

    # left column: category, description, how often it occurs
    
    freq = 'EGD {:.1f}%, UTSW-Glioma {:.1f}% of test patients.'.format(table2.loc[category, 'EGD, % of test patients'],
                                                                     table2.loc[category, 'UTSW-Glioma, % of test patients'])
                                                                     
    ax = fig.add_subplot(gs[i, 0])
    ax.axis('off')
    
    if category == 'S1 Confident and correct':
        ax.text(0.0, 0.66, letter + '.  ' + textwrap.fill("Correct, single subtype", 24, subsequent_indent='    '), fontsize=7.5, fontweight='bold', va='bottom')
    elif category == 'S2 Borderline call, ambiguity flagged':
        ax.text(0.0, 0.60, letter + '.  ' + textwrap.fill("Borderline call, uncertainty flagged", 24, subsequent_indent='    '), fontsize=7.5, fontweight='bold', va='bottom')
    else:
        ax.text(0.0, 0.66, letter + '.  ' + textwrap.fill(category[3:], 24, subsequent_indent='    '), fontsize=7.5, fontweight='bold', va='bottom')

    ax.text(0.0, 0.60, textwrap.fill(description, 30) + '\n\n' + textwrap.fill(freq, 30), fontsize=6.6, va='top', color='#333333')

    # contrast-enhanced T1
    ax = fig.add_subplot(gs[i, 1])
    ax.imshow(t1ce_crop, cmap='gray', vmin=np.percentile(t1ce_slice[brain_slice], 1), vmax=np.percentile(t1ce_slice[brain_slice], 99.5),
              interpolation='lanczos')
    ax.contour(mask_crop, levels=[0.5], colors='#FFD400', linewidths=0.7)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_ylabel(case, fontsize=7, fontweight='bold')
    if i == 0:
        ax.set_title('Contrast-enhanced T1', fontsize=8, fontweight='bold', pad=4)

    # FLAIR
    ax = fig.add_subplot(gs[i, 2])
    ax.imshow(flair_crop, cmap='gray', vmin=np.percentile(flair_slice[brain_slice], 1), vmax=np.percentile(flair_slice[brain_slice], 99.5),
              interpolation='lanczos')
    ax.contour(mask_crop, levels=[0.5], colors='#FFD400', linewidths=0.7)
    ax.set_xticks([])
    ax.set_yticks([])
    if i == 0:
        ax.set_title('FLAIR', fontsize=8, fontweight='bold', pad=4)

    # right column: probabilities, argmax, 90% set
    set_names = ', '.join(full_name[s.strip()] for s in r['modal_set'].strip('{}').split(',')) if r['modal_set'] != '{}' else ''
    text = ('Cohort: ' + ('EGD' if case.startswith('EGD') else 'UTSW-Glioma') +
            '\nMolecular truth: ' + full_name[r['true']] +
            '\n\nModel probabilities: A {:.2f}, G {:.2f}, O {:.2f}'.format(r['pA'], r['pG'], r['pO']) +
            '\n\nArgmax: ' + full_name[r['argmax']] + ('' if r['argmax_correct'] else ' (wrong)') +
            '\n90% set: ' + textwrap.fill('{' + set_names + '}', 30, subsequent_indent='   ') +
            '\nSame set in {:.0f}% of test-half splits'.format(100 * r['set_stability']))
    ax = fig.add_subplot(gs[i, 3])
    ax.axis('off')
    ax.text(0.04, 0.5, text, fontsize=6.8, va='center', linespacing=1.35)

for name in ['Figure5', 'Figure6']:
    figs[name].savefig(os.path.join(OUT, name + '.png'), dpi=600)
    #figs[name].savefig(os.path.join(OUT, name + '.tiff'), dpi=1200, pil_kwargs={'compression': 'tiff_lzw'})
    plt.close(figs[name])
print('saved Figure5 and Figure6')


findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberati

saved Figure5 and Figure6
